# 🔥 MODO BESTIA: Entrenamiento de IA de Video para Google Veo 3.1 y Higgsfield
### Modelo Base: Meta-Llama-3.1-8B-Instruct | LoRA Rank = 32 | Contexto = 2048 Tokens
### Optimizado para exprimir el 85% de la GPU T4 de Google Colab (~12.5 GB VRAM)

Este cuaderno entrena a **Llama 3.1 (8B)** para que actúe como un Director de Cine e Image-to-Video (I2V):
1. **Preserva el ancla al 100%** (elimina el derretimiento facial / *morphing*).
2. **Estructura en 3 bloques obligatorios:** `CÁMARA con inercia` + `SUJETO con micro-acciones` + `ENTORNO constante con etiqueta Audio:`.
3. **Castiga los clichés de IA:** Cero *cinematic zoom*, cero *8k*, cero movimientos bruscos.

## Paso 1: Instalación de Unsloth y Librerías de Alto Rendimiento
Unsloth permite entrenar un modelo de **8.000 millones de parámetros** dentro de los 15 GB de la T4 gratuita sin desbordar memoria.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

## Paso 2: Cargar Meta-Llama-3.1-8B-Instruct (Cuantización 4-bit)
Cargamos el titán de 8B con contexto ampliado a 2048 tokens.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Modo Bestia: contexto amplio para tomas complejas
dtype = None  # Detección automática (Float16)
load_in_4bit = True  # Cuantización de 4 bits para mantener los pesos en ~5.5 GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✅ Llama-3.1-8B cargado exitosamente en la GPU.")

## Paso 3: Configurar LoRA de Alta Capacidad (Modo Bestia: r = 32, alpha = 64)
Duplicamos la capacidad del adaptador LoRA para capturar todas las sutilezas de la cinemática de Veo 3.1.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,            # Duplicado de 16 a 32 (el doble de capacidad representacional)
    lora_alpha=64,   # Escalado proporcional
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Ahorro masivo de memoria VRAM
    random_state=3407,
)

print("✅ Adaptadores LoRA r=32 configurados en Modo Bestia.")

## Paso 4: Cargar el Dataset de Video (`dataset_video_dpo.jsonl`)
Sube el archivo `dataset_video_dpo.jsonl` generado para entrenar con preferencia DPO.

In [ ]:
import os
import json
from google.colab import files
from datasets import Dataset

if not os.path.exists("dataset_video_dpo.jsonl"):
    print("Por favor, sube el archivo 'dataset_video_dpo.jsonl':")
    uploaded = files.upload()

data_list = []
with open("dataset_video_dpo.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        data_list.append({
            "prompt": item["prompt"],
            "chosen": item["chosen"],
            "rejected": item["rejected"]
        })

dataset = Dataset.from_list(data_list)
print(f"✅ Dataset de Video cargado: {len(dataset)} pares de entrenamiento DPO.")
print("Ejemplo:\nChosen:", dataset[0]['chosen'][:160], "...")

## Paso 5: Entrenamiento DPO Maximizando VRAM (~12.5 GB)
Configuramos `DPOTrainer` con `per_device_train_batch_size = 2` y `gradient_accumulation_steps = 4` para exprimir la memoria de la T4 sin desbordar.

In [ ]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_config = DPOConfig(
    output_dir="./resultados_video_bestia",
    beta=0.1,  # Factor de castigo para clichés de IA y re-descripción facial
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    warmup_ratio=0.1,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    save_strategy="no",
    seed=3407,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_length=max_seq_length,
    max_prompt_length=512,
)

# Monitoreo de memoria VRAM antes de arrancar
gpu_mem_used = torch.cuda.memory_allocated() / 1024**3
gpu_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"📊 VRAM Inicial: {gpu_mem_used:.2f} GB / {gpu_total:.2f} GB")

print("🔥 Iniciando entrenamiento Modo Bestia (Llama-3.1-8B)... Duración aprox: 18-22 minutos.")
stats = dpo_trainer.train()

gpu_mem_peak = torch.cuda.max_memory_allocated() / 1024**3
print(f"🎉 ¡Entrenamiento completado! VRAM Máxima aprovechada: {gpu_mem_peak:.2f} GB / {gpu_total:.2f} GB")

## Paso 6: Prueba en Vivo de Inferencia (Director de Video Veo 3.1)
Probamos la IA de 8B con un caso nuevo para generar el prompt de 3 bloques listo para Google Flow.

In [ ]:
FastLanguageModel.for_inference(model)

def generar_prompt_video_veo(ancla_desc, accion_deseada):
    prompt_in = f"Generate an Image-to-Video (I2V) prompt for Veo 3.1 based on this anchor: {ancla_desc}. Action: {accion_deseada}"
    inputs = tokenizer([prompt_in], return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1
    )
    res = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if prompt_in in res:
        res = res.replace(prompt_in, "").strip()
    return res

# Prueba
ancla_ejemplo = "Fotografía ancla de un pescador en un muelle con impermeable amarillo y barba mojada"
accion_ejemplo = "El pescador exhala vaho frío y mira fijamente al horizonte mientras la lluvia cae"

prompt_veo = generar_prompt_video_veo(ancla_ejemplo, accion_ejemplo)
print("=== PROMPT DE VIDEO GENERADO PARA VEO 3.1 ===\n")
print(prompt_veo)

## Paso 7: Exportar y Descargar los Pesos Entrenados de 8B
Empaquetamos el adaptador LoRA entrenado en un archivo zip para descargarlo a tu equipo.

In [ ]:
model.save_pretrained_merged("lora_video_modo_bestia_8b", tokenizer, save_method="lora")
print("✅ Pesos LoRA de 8B guardados exitosamente.")

!zip -r lora_video_modo_bestia_8b.zip lora_video_modo_bestia_8b
files.download("lora_video_modo_bestia_8b.zip")
print("💾 Descarga iniciada a tu computadora.")